# Chunking Strategies
> Compare six built-in chunkers and learn when to use each one.

Anchor ships with chunkers optimized for different content types: plain text,
semantic boundaries, sentences, source code, and tabular data.

**Time:** ~10 minutes

## Setup

In [ ]:
from anchor.ingestion.chunkers import (
    FixedSizeChunker,
    RecursiveCharacterChunker,
    SemanticChunker,
    SentenceChunker,
    CodeChunker,
    TableAwareChunker,
)

## Sample Texts
We prepare different content types to demonstrate each chunker's strengths.

In [ ]:
plain_text = (
    "Anchor is a framework for building context-aware AI applications. "
    "It provides tools for memory management, retrieval-augmented generation, "
    "and intelligent context assembly. The framework is designed to be modular "
    "and extensible. Developers can pick and choose the components they need. "
    "The ingestion module handles converting raw documents into structured "
    "context items. It supports multiple chunking strategies optimized for "
    "different content types. Text is split into overlapping chunks to preserve "
    "context at boundaries. Once ingested, context items flow through the "
    "pipeline where they are scored, filtered, and assembled into the prompt."
)

code_text = '''def fibonacci(n):
    """Return the nth Fibonacci number."""
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

def factorial(n):
    """Return n factorial."""
    if n <= 1:
        return 1
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

def is_prime(n):
    """Check if n is a prime number."""
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True
'''

table_text = (
    "| Language | Year | Typing   |\n"
    "|----------|------|----------|\n"
    "| Python   | 1991 | Dynamic  |\n"
    "| Java     | 1995 | Static   |\n"
    "| Rust     | 2010 | Static   |\n"
    "| Go       | 2009 | Static   |\n"
    "\n"
    "These languages represent different design philosophies. "
    "Python favors readability. Java emphasizes enterprise patterns. "
    "Rust prioritizes memory safety. Go targets simplicity and concurrency."
)

print(f"Plain text: {len(plain_text)} chars")
print(f"Code text:  {len(code_text)} chars")
print(f"Table text: {len(table_text)} chars")

## 1. FixedSizeChunker
Splits text into chunks of exactly `chunk_size` characters with a sliding
overlap. Simple and predictable.

In [ ]:
fixed = FixedSizeChunker(chunk_size=200, overlap=50)
chunks = fixed.chunk(plain_text)

print(f"FixedSizeChunker: {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] ({len(chunk)} chars): {chunk[:60]}...")

## 2. RecursiveCharacterChunker
Tries to split on paragraph breaks first, then sentences, then words.
Produces more natural chunk boundaries than fixed-size.

In [ ]:
recursive = RecursiveCharacterChunker(chunk_size=200, overlap=50)
chunks = recursive.chunk(plain_text)

print(f"RecursiveCharacterChunker: {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] ({len(chunk)} chars): {chunk[:60]}...")

## 3. SemanticChunker
Groups sentences whose embeddings are above a similarity threshold.
Requires an embedding function.

In [ ]:
# Mock embedding function for demonstration
embed_fn = lambda text: [hash(c) % 100 / 100.0 for c in text[:64].ljust(64)]

semantic = SemanticChunker(
    embed_fn=embed_fn,
    threshold=0.5,
    chunk_size=200,
)
chunks = semantic.chunk(plain_text)

print(f"SemanticChunker: {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] ({len(chunk)} chars): {chunk[:60]}...")

## 4. SentenceChunker
Splits on sentence boundaries. The `overlap` parameter controls how many
trailing sentences from the previous chunk are prepended to the next.

In [ ]:
sentence = SentenceChunker(chunk_size=200, overlap=1)
chunks = sentence.chunk(plain_text)

print(f"SentenceChunker: {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] ({len(chunk)} chars): {chunk[:60]}...")

## 5. CodeChunker
Splits source code at function/class boundaries. Keeps logical units
together.

In [ ]:
code_chunker = CodeChunker(chunk_size=200, overlap=50)
chunks = code_chunker.chunk(code_text)

print(f"CodeChunker: {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    lines = chunk.strip().split('\n')
    print(f"  [{i}] ({len(chunk)} chars, {len(lines)} lines):")
    for line in lines[:3]:
        print(f"      {line}")
    if len(lines) > 3:
        print(f"      ... (+{len(lines) - 3} more lines)")
    print()

## 6. TableAwareChunker
Detects tables (Markdown or delimited) and keeps them intact rather
than splitting mid-row.

In [ ]:
table_chunker = TableAwareChunker(chunk_size=200, overlap=100)
chunks = table_chunker.chunk(table_text)

print(f"TableAwareChunker: {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"  [{i}] ({len(chunk)} chars):")
    for line in chunk.strip().split('\n')[:5]:
        print(f"      {line}")
    print()

## Side-by-Side Comparison on Plain Text

In [ ]:
chunkers = {
    "FixedSize":     FixedSizeChunker(chunk_size=200, overlap=50),
    "Recursive":     RecursiveCharacterChunker(chunk_size=200, overlap=50),
    "Semantic":      SemanticChunker(embed_fn=embed_fn, threshold=0.5, chunk_size=200),
    "Sentence":      SentenceChunker(chunk_size=200, overlap=1),
    "Code":          CodeChunker(chunk_size=200, overlap=50),
    "TableAware":    TableAwareChunker(chunk_size=200, overlap=100),
}

print(f"{'Chunker':<16} {'Chunks':>6}  {'Avg Size':>8}")
print("-" * 34)
for name, chunker in chunkers.items():
    chunks = chunker.chunk(plain_text)
    avg = sum(len(c) for c in chunks) / len(chunks) if chunks else 0
    print(f"{name:<16} {len(chunks):>6}  {avg:>8.0f}")

## Key Takeaways
- **FixedSizeChunker**: predictable sizing, may break mid-word
- **RecursiveCharacterChunker**: best general-purpose choice for prose
- **SemanticChunker**: groups semantically related sentences (needs embeddings)
- **SentenceChunker**: respects sentence boundaries, good for Q&A
- **CodeChunker**: preserves function/class boundaries in source code
- **TableAwareChunker**: keeps tables intact, ideal for structured data
- All chunkers share the same `.chunk(text) -> list[str]` interface

**Next:** [Parsers](03_parsers.ipynb)